In [4]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader

C:\Users\suhan\AppData\Local\Temp\ipykernel_6108\161500683.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [5]:
load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")

Loading Data

In [6]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")

text_loader = TextLoader(DATA_FILE_PATH, encoding= "utf-8")

documents = text_loader.load()


print("DATA LOADED")
print("="*40)
print(documents)

DATA LOADED
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

In [8]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [9]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


In [14]:
print(f"Total characters in document: {len(documents[0].page_content)}")

Total characters in document: 2597


Splitting Data


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = text_splitter.split_documents(documents)
print(chunks)
print("-"*50)
print(len(chunks))

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [ ]:
print(chunks[5])

page_content='5. REIMBURSEMENT POLICY
Employees can claim reimbursement for approved business expenses such as travel,
client meals, and internet bills used for official work.
All reimbursement claims must be submitted with valid bills within 30 days of the expense.
Claims are processed within 10 working days after approval from the reporting manager.' metadata={'source': 'data\\hr_policy.txt'}


Embedding Data

In [11]:
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(model_name="jina-clip-v2")

print("Embedding model is ready. Model Name - ", embeddings_model.model_name)

Embedding model is ready. Model Name -  jina-clip-v2


Store Data in Vectore DB

In [12]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embedding = embeddings_model)

print(f"{vector_store.index.ntotal} Chunks are stored")

9 Chunks are stored


In [13]:
test_query = "How many sick leaves employee gets"

top_matches = vector_store.similarity_search(test_query, k=2)

for i, match in enumerate(top_matches, start=1):
    print(f"----- match {i} ----")
    print(match.page_content)
    print()

----- match 1 ----
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

----- match 2 ----
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



TOOL

In [14]:
retriever = vector_store.as_retriever(search_kwargs = {"k":3})

def seach_hr_policy(question: str) -> str:
    """Search the HR policy document for information about leave, work from home, probation, notice period, reimbursement, 
    code of conduct holidays, or exit process"""
    maching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in maching_chunks)

LLM

In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-20b",
    temperature=0.5
)

test_response = llm.invoke("Explain me working of RAG in 100 words")
print(test_response.content)

Retrieval‑Augmented Generation (RAG) combines a language model with an external knowledge base. When a query arrives, a retrieval component first searches indexed documents or embeddings to fetch the most relevant passages. These retrieved snippets are then fed, along with the original prompt, into a generative model such as GPT. The model conditions its output on both the prompt and the evidence, producing answers that are grounded in up‑to‑date or domain‑specific information. RAG thus improves factual accuracy, reduces hallucinations, and allows dynamic updates without retraining the entire model. RAG can adapt to facts, making it ideal for like medicine or finance.


In [15]:
from langchain.agents import create_agent

hr_assistant = create_agent(
    model=llm,
    tools=[seach_hr_policy],
    system_prompt=
    """
    You are a friendly HR Assistant. Always use search_hr_policy tool to look up for the facts before answering.
    If the answer isn't in the search results say you don't know instead of guessing.
    """
)

print("HR Assistant is ready to answer your queries!")

HR Assistant is ready to answer your queries!


In [16]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and printa nicely formatted answer."""
    print("="*50)
    print("QUESTION - ", question)
    print("="*50)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER - ", answer)
    print("="*60, "\n")
    return answer 


In [17]:
response = ask_hr_assistant("tell me about leave policies")

QUESTION -  tell me about leave policies
ANSWER -  Sure thing! Here’s a quick rundown of our leave policy:

| **Type of Leave** | **Entitlement** | **Key Rules** |
|-------------------|-----------------|---------------|
| **Annual (Paid) Leave** | 20 days per calendar year (full‑time employees) | • Submit requests via the HR portal **at least 5 working days in advance**.<br>• Unused days can be carried forward, but only up to **5 days** into the next year. |
| **Sick Leave** | 10 paid days per year | • Separate from annual leave.<br>• A medical certificate is required for sick leave lasting **more than 2 consecutive days**. |

**Quick Tips**

- **Requesting Leave:** Log into the HR portal, fill out the leave request form, and hit submit. The system will automatically notify your manager and HR.
- **Carry‑over:** If you have unused annual leave, you can roll over up to 5 days. Anything beyond that will be forfeited.
- **Sick Leave Documentation:** For any sick leave longer than 2 days, 